In [0]:
import time
from pyspark.sql import functions as F

base = spark.table("workspace.olist_nuevo.gold_ventas")
print("Filas base:", base.count())

def replicar(df, factor):
    reps = spark.range(factor).withColumnRenamed("id", "rep")
    return (df.crossJoin(reps)
              .withColumn("order_id", F.concat_ws("-", "order_id", F.col("rep").cast("string")))
              .drop("rep")
              .withColumn("anio", F.year("order_purchase_timestamp"))
              .withColumn("mes", F.month("order_purchase_timestamp")))

tiempos_carga = []

def crear_tablas(factor):
    df = replicar(base, factor)
    t0 = time.time()
    df.write.mode("overwrite").saveAsTable(f"workspace.olist_nuevo.exp_ventas_x{factor}")
    t1 = time.time()
    df.write.mode("overwrite").partitionBy("anio", "mes") \
      .saveAsTable(f"workspace.olist_nuevo.exp_ventas_x{factor}_part")
    t2 = time.time()
    tiempos_carga.append((factor, round(t1 - t0, 1), round(t2 - t1, 1)))
    print(f"Factor x{factor}: listo")

Filas base: 108638


In [0]:
for factor in [1, 10]:
    crear_tablas(factor)

Factor x1: listo
Factor x10: listo


In [0]:
crear_tablas(50)

Factor x50: listo


In [0]:
display(spark.createDataFrame(tiempos_carga,
        ["factor", "carga_sin_particion_s", "carga_particionada_s"]))

factor,carga_sin_particion_s,carga_particionada_s
1,12.4,5.2
10,4.4,5.2
50,8.1,8.0


In [0]:
consultas = {
    "ventas_por_mes": "SELECT anio, mes, ROUND(SUM(price),2) ventas FROM {t} GROUP BY anio, mes ORDER BY anio, mes",
    "top_categorias": "SELECT product_category_name_english categoria, ROUND(SUM(price),2) ventas FROM {t} GROUP BY 1 ORDER BY ventas DESC LIMIT 10",
    "ventas_por_estado": "SELECT customer_state, COUNT(DISTINCT order_id) pedidos, ROUND(SUM(price),2) ventas FROM {t} GROUP BY 1 ORDER BY ventas DESC",
    "filtro_periodo": "SELECT COUNT(*) filas, ROUND(SUM(price),2) ventas FROM {t} WHERE anio = 2018 AND mes = 1"
}

def medir(tabla, repeticiones=3):
    filas = []
    for nombre, sql in consultas.items():
        tiempos = []
        for _ in range(repeticiones):
            t0 = time.time()
            spark.sql(sql.format(t=tabla)).collect()
            tiempos.append(time.time() - t0)
        tiempos.sort()
        filas.append((tabla.split(".")[-1], nombre, round(tiempos[len(tiempos)//2], 2)))
    return filas

In [0]:
resultados = []
for factor in [1, 10, 50]:
    for sufijo in ["", "_part"]:
        resultados += medir(f"workspace.olist_nuevo.exp_ventas_x{factor}{sufijo}")

df_res = spark.createDataFrame(resultados, ["tabla", "consulta", "mediana_segundos"])

columnas = ["exp_ventas_x1", "exp_ventas_x1_part",
            "exp_ventas_x10", "exp_ventas_x10_part",
            "exp_ventas_x50", "exp_ventas_x50_part"]

display(
    df_res
    .groupBy("consulta")
    .pivot("tabla", columnas)
    .agg(F.first("mediana_segundos"))
    .orderBy("consulta")
)/Volumes/workspace/olist_nuevo/olist/Archivos.csv/Olist_CSV/

consulta,exp_ventas_x1,exp_ventas_x1_part,exp_ventas_x10,exp_ventas_x10_part,exp_ventas_x50,exp_ventas_x50_part
filtro_periodo,0.57,0.55,0.57,0.53,0.61,0.53
top_categorias,0.59,0.63,0.59,0.61,0.65,0.62
ventas_por_estado,0.62,0.68,0.74,0.78,1.23,1.14
ventas_por_mes,0.64,0.67,0.57,0.62,0.64,0.62


Databricks visualization. Run in Databricks to view.

In [0]:
filas = []
for factor in [1, 10, 50]:
    for sufijo in ["", "_part"]:
        d = spark.sql(f"DESCRIBE DETAIL workspace.olist_nuevo.exp_ventas_x{factor}{sufijo}").first()
        filas.append((f"exp_ventas_x{factor}{sufijo}", d["numFiles"], d["sizeInBytes"],
                      round(d["sizeInBytes"] / 1048576, 1), str(d["partitionColumns"])))

display(spark.createDataFrame(filas,
        ["tabla", "numFiles", "sizeInBytes", "tamano_MB", "particiones"]))

tabla,numFiles,sizeInBytes,tamano_MB,particiones
exp_ventas_x1,1,8266791,7.9,[]
exp_ventas_x1_part,23,10132485,9.7,"['anio', 'mes']"
exp_ventas_x10,1,14934164,14.2,[]
exp_ventas_x10_part,23,17404523,16.6,"['anio', 'mes']"
exp_ventas_x50,1,44437298,42.4,[]
exp_ventas_x50_part,23,25767449,24.6,"['anio', 'mes']"


Databricks visualization. Run in Databricks to view.